# Hy-MT2-30B-A3B-uncensored-heretic — Colab L4 一鍵自動化翻譯（EN -> 台繁成人向）

**使用步驟：**
1. 確認右上角硬體加速器為 **L4 GPU**（24 GB VRAM；若配到 T4 會自動拒跑並釋放資源）。
2. 確認本機 Agent 已透過 `drive_bridge.py push` 將 `<job>.jsonl` 與 `.manifest.json` 推入 `MyDrive/galgame_mtl/hymt2/input/`。
3. 按 **`Ctrl+F9`（全部執行）**，點「允許」授權 Google Drive 後即可離開電腦。
4. 結束碼區分：`rc = 0`（`[DONE]`：寫入 `.DONE.json`；翻不出而保留英文原文的列數不超過總列數 0.1%，摘要會顯示待檢查數量）；`rc = 2`（`[PARTIAL]`：全部已跑完並存檔，但保留原文的列數超過 0.1% 等——**不跳紅字、不寫 `.DONE.json`、同步 Drive 後照 `AUTO_DISCONNECT` 自動釋放 L4 省點數**）；`rc = 1`（啟動 Gate／硬體／崩潰等真正異常才拋出 `RuntimeError` 並依 `DISCONNECT_ON_ERROR=False` 保留 L4 最多 `IDLE_UNASSIGN_MINUTES=15` 分鐘除錯）。

In [ ]:
#@title 1. 執行參數 { display-mode: "form" }
AUTO_DISCONNECT = True #@param {type:"boolean"}
DISCONNECT_ON_ERROR = False #@param {type:"boolean"}
IDLE_UNASSIGN_MINUTES = 15 #@param {type:"integer"}
TW_CONVERT_MODE = "s2twp" #@param ["s2twp", "s2t"]
REPAIR_ONLY = False #@param {type:"boolean"}
TARGET_JOB = "" #@param {type:"string"}
FORCE_RERUN = False #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/galgame_mtl" #@param {type:"string"}
REPO_URL = "https://github.com/poynt2005/hanhua-mtl.git" #@param {type:"string"}
BRANCH = "hymt2-heretic-colab" #@param {type:"string"}

In [ ]:
#@title 2. 一鍵啟動 GPU 4 道驗活 + 4 路並行翻譯 + s2twp QA + 自動斷線
from collections import deque
import shutil
import subprocess
import sys
import time
from pathlib import Path
from google.colab import drive, runtime

def _load_runner_dir(drive_root: Path) -> Path:
    drive_scripts = drive_root / "scripts" / "hymt2"
    if (drive_scripts / "runner.py").is_file() and (drive_scripts / "schema.py").is_file():
        local_scripts = Path("/content/_runner_hymt2")
        local_scripts.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(drive_scripts / "schema.py", local_scripts / "schema.py")
        shutil.copyfile(drive_scripts / "runner.py", local_scripts / "runner.py")
        print(f"[init] 已從 Drive 載入腳本: {drive_scripts}", flush=True)
        return local_scripts

    repo_dir = Path("/content/_repo")
    if repo_dir.exists():
        shutil.rmtree(repo_dir, ignore_errors=True)
    print(f"[init] Drive 無同步腳本，改由 Git 取得 {BRANCH}...", flush=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)],
        check=True,
    )
    return repo_dir / "deploy" / "colab-hymt2"

mounted = False
rc = 1
try:
    drive.mount("/content/drive", force_remount=False)
    mounted = True
    pip_res = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "opencc-python-reimplemented"],
        capture_output=True,
        text=True,
    )
    if pip_res.returncode != 0:
        raise RuntimeError(f"pip install opencc failed:\n{pip_res.stdout}\n{pip_res.stderr}")
    runner_dir = _load_runner_dir(Path(DRIVE_ROOT))
    work_dir = Path(DRIVE_ROOT) / "hymt2" / "work"
    work_dir.mkdir(parents=True, exist_ok=True)
    runner_log = work_dir / "runner.log"
    cmd = [
        sys.executable,
        "-u",
        str(runner_dir / "runner.py"),
        "--drive-root", DRIVE_ROOT,
        "--tw-mode", TW_CONVERT_MODE,
    ]
    if FORCE_RERUN:
        cmd.append("--force")
    if REPAIR_ONLY:
        cmd.append("--repair-only")
    if TARGET_JOB.strip():
        cmd.extend(["--job", TARGET_JOB.strip()])
    tail = deque(maxlen=30)
    with open(runner_log, "w", encoding="utf-8", newline="\n") as log_fh:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            log_fh.write(line)
            log_fh.flush()
            tail.append(line.rstrip("\r\n"))
        rc = proc.wait()
    if rc == 0:
        print("\n[DONE] 翻譯任務已完成並寫入 .DONE.json。請看上方摘要的「待你檢查」列數（清單在 hymt2/output/<job>.review.jsonl）。", flush=True)
    elif rc == 2:
        print("\n[PARTIAL] ============================================================", flush=True)
        print("[PARTIAL] 全部句子已跑完，譯文輸出、checkpoint.jsonl、qa_report.json 與 review.jsonl 皆已寫入。", flush=True)
        print("[PARTIAL] 待補列數超過上限（或有缺漏），故未寫入 .DONE.json（不會跳錯也不保留待機燒點數）。", flush=True)
        print("[PARTIAL] 下一步：在本機執行 drive_bridge.py status 檢視 hymt2/output/<job>.review.jsonl（模型原始回覆在 hymt2/work/<job>.attempts.jsonl），填入 translation 後另存為 overrides.jsonl，或用 REPAIR_ONLY=True 重跑。", flush=True)
        print("[PARTIAL] ============================================================", flush=True)
    else:
        tail_txt = "\n".join(tail)
        raise RuntimeError(
            f"runner.py 結束碼 {rc} (完整日誌已存至 hymt2/work/runner.log)。最後輸出：\n{tail_txt}"
        )
finally:
    if mounted:
        try:
            print("[flush] 正在將輸出與日誌同步回 Google Drive...")
            drive.flush_and_unmount()
            print("[flush] Drive 同步完成。")
        except Exception as exc:
            print(f"[flush] warn: {exc}")
    should_disconnect = (rc in (0, 2) and AUTO_DISCONNECT) or (rc not in (0, 2) and DISCONNECT_ON_ERROR)
    if should_disconnect:
        print("[power] 正在釋放 Colab GPU 執行階段...")
        runtime.unassign()
    elif rc in (0, 2):
        print("[power] AUTO_DISCONNECT=False：請記得手動中斷執行階段以免持續計費。", flush=True)
    elif rc not in (0, 2):
        print("[power] ============================================================", flush=True)
        print("[power] STOP THE RUNTIME NOW (若不繼續除錯請立即手動中斷執行階段！)", flush=True)
        print("[power] DISCONNECT_ON_ERROR=False：已保留 L4 與 /content/models/（Colab L4 待機約 ~2 compute units/hr）。", flush=True)
        if IDLE_UNASSIGN_MINUTES > 0:
            print(f"[power] {IDLE_UNASSIGN_MINUTES} 分鐘內無動作將自動釋放 GPU（按停止/中斷鍵可保留除錯）。", flush=True)
            try:
                time.sleep(IDLE_UNASSIGN_MINUTES * 60)
                print("[power] 已達待機上限，自動釋放 Colab GPU 執行階段...", flush=True)
                runtime.unassign()
            except KeyboardInterrupt:
                print("[power] 已取消自動釋放倒數（請記得除錯後手動 STOP THE RUNTIME NOW）。", flush=True)
